# Agent handoff: one bounded evidence contract per surfaced patient

Turns the gold tables into the JSON envelope the referral agent is allowed to see.

| | |
| --- | --- |
| **Reads** | `gold_referral_state`, `gold_criteria_hits`, `gold_evidence`, `bronze_reference_release` |
| **Writes** | `gold_agent_contracts`, plus one JSON file per patient under `Files/contracts/` |

## Why a contract rather than a query

The agent could be given a SQL endpoint and told to look things up. It is given a fixed
envelope instead, and the difference is the whole safety argument:

* **It cannot reach a patient it was not handed.** Scope is a property of the input, not
  of the agent behaving.
* **Every value it can state is already an identified row.** There is nothing in the
  envelope without an `evidence_id`, so a claim with no citation is a claim about
  something that was never supplied.
* **The run is reproducible.** The same envelope produces a comparable brief next month.
  A live query does not, because the record underneath has moved.

Only patients in state `indicators_present` get a contract. A patient the pipeline did
not surface is a patient the agent must not write about, and the cleanest way to enforce
that is to never hand it over.

## The `family_history_status` field

Carried explicitly rather than left to be inferred from missing keys. `never_taken`
means nobody asked, and the agent's clinical gate checks that the brief says so instead
of describing the history as clear.

In [ ]:
MAX_CONTRACTS = 40
MAX_EVIDENCE_PER_PATIENT = 60
PIPELINE_RUN_ID = ""

In [ ]:
import json

import notebookutils
from pyspark.sql import functions as F

RUN_ID = PIPELINE_RUN_ID or "local"

_WS = notebookutils.runtime.context["currentWorkspaceId"]
_ONELAKE = notebookutils.conf.get("trident.onelake.endpoint").replace("https://", "")
_LAKEHOUSE_ID = {}


def lake_table(lakehouse, table):
    """Read a table from a lakehouse other than the attached default.

    spark.read.table() resolves only against the default lakehouse, so a cross-lakehouse
    read has to go by OneLake path -- and that path needs the lakehouse *id*. Mixing the
    workspace id with the lakehouse *name* returns a 400 from the ABFS driver, which
    surfaces as an opaque Py4JJavaError rather than anything mentioning names.
    """
    if lakehouse not in _LAKEHOUSE_ID:
        _LAKEHOUSE_ID[lakehouse] = notebookutils.lakehouse.get(
            lakehouse, workspaceId=_WS).id
    return spark.read.format("delta").load(
        f"abfss://{_WS}@{_ONELAKE}/{_LAKEHOUSE_ID[lakehouse]}/Tables/{table}")


state = spark.table("gold_referral_state")
hits = spark.table("gold_criteria_hits")
evidence = spark.table("gold_evidence")
release = lake_table("bronze_lakehouse", "bronze_reference_release").collect()[0]
history = lake_table("silver_lakehouse", "silver_family_history")

surfaced = state.filter("referral_state = 'indicators_present'")
print(f"surfaced patients: {surfaced.count():,}")
print(f"reference release: {release['source']} read {release['read_on']}")

In [ ]:
# --------------------------------------------- choose a demonstrable spread
# Not the top N by anything. A demo needs a patient surfaced by one sufficient
# criterion, one surfaced only by combination, and one whose family history was never
# taken -- the three shapes a clinician will actually meet.
ranked = (surfaced.join(history.select("patient_id", "history_taken"),
                        "patient_id", "left")
          .withColumn("history_taken", F.coalesce("history_taken", F.lit(False)))
          .orderBy(F.desc("sufficient_fired"), F.desc("criteria_fired"),
                   "patient_id"))

by_sufficient = ranked.filter("sufficient_fired > 0").limit(MAX_CONTRACTS // 2)
by_combination = (ranked.filter("sufficient_fired = 0 AND contributory_fired >= 2")
                  .limit(MAX_CONTRACTS // 4))
no_history = (ranked.filter("history_taken = false")
              .limit(MAX_CONTRACTS // 4))

chosen = (by_sufficient.unionByName(by_combination).unionByName(no_history)
          .dropDuplicates(["patient_id"]).limit(MAX_CONTRACTS))
chosen_ids = [row["patient_id"] for row in chosen.select("patient_id").collect()]

print(f"contracts to publish: {len(chosen_ids)}")
print(f"  surfaced by a sufficient criterion : {by_sufficient.count()}")
print(f"  surfaced only by combination       : {by_combination.count()}")
print(f"  with no family history ever taken  : {no_history.count()}")

In [ ]:
# ------------------------------------------------------- build the envelopes
state_rows = {r["patient_id"]: r for r in chosen.collect()}
hit_rows = {}
for row in hits.filter(F.col("patient_id").isin(chosen_ids)).collect():
    hit_rows.setdefault(row["patient_id"], []).append(row)
evidence_rows = {}
for row in evidence.filter(F.col("patient_id").isin(chosen_ids)).collect():
    evidence_rows.setdefault(row["patient_id"], []).append(row)

contracts = []
for patient_id in chosen_ids:
    record = state_rows[patient_id]
    items = sorted(evidence_rows.get(patient_id, []),
                   key=lambda r: (r["evidence_date"] or "", r["evidence_id"]))

    # A bounded envelope. If a patient somehow carries more evidence than the agent can
    # reasonably read, truncate deliberately and SAY SO in the contract, rather than
    # letting the model quietly see a subset it believes is complete.
    truncated = len(items) > MAX_EVIDENCE_PER_PATIENT
    if truncated:
        items = items[-MAX_EVIDENCE_PER_PATIENT:]

    contracts.append({
        "patient": {
            "patient_id": patient_id,
            "referral_state": record["referral_state"],
            "age_years": int(record["age_years"]) if record["age_years"] is not None
                         else None,
            "family_history_status": ("taken" if record["history_taken"]
                                      else "never_taken"),
        },
        "criteria": [
            {"criterion": h["criterion"], "tier": h["tier"],
             "description": h["criterion_description"]}
            for h in sorted(hit_rows.get(patient_id, []),
                            key=lambda r: (r["tier"] != "sufficient", r["criterion"]))
        ],
        "evidence": [
            {"evidence_id": item["evidence_id"],
             "evidence_type": item["evidence_type"],
             "evidence_date": item["evidence_date"],
             "evidence_text": item["evidence_text"]}
            for item in items
        ],
        "provenance": {
            "run_id": RUN_ID,
            "reference_source": release["source"],
            "reference_read_on": str(release["read_on"]),
            "evidence_truncated": truncated,
            "note": ("All data is synthetic. No genomic data is involved. This is a "
                     "case-finding output for clinician review, not a referral "
                     "decision."),
        },
    })

print(f"built {len(contracts)} contracts")
sizes = sorted(len(c["evidence"]) for c in contracts)
print(f"evidence per contract: min={sizes[0]}  median={sizes[len(sizes) // 2]}  "
      f"max={sizes[-1]}")

In [ ]:
# ------------------------------------------------------------------- persist
import os

directory = (f"/lakehouse/default/Files/contracts")
os.makedirs(directory, exist_ok=True)
for contract in contracts:
    patient_id = contract["patient"]["patient_id"]
    with open(f"{directory}/patient-evidence.{patient_id}.json", "w",
              encoding="utf-8") as handle:
        json.dump(contract, handle, indent=2)
print(f"wrote {len(contracts)} files to Files/contracts/")

rows = [{"patient_id": c["patient"]["patient_id"],
         "referral_state": c["patient"]["referral_state"],
         "criteria_count": len(c["criteria"]),
         "evidence_count": len(c["evidence"]),
         "family_history_status": c["patient"]["family_history_status"],
         "contract_json": json.dumps(c),
         "run_id": RUN_ID}
        for c in contracts]
spark.createDataFrame(rows).write.mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold_agent_contracts")
print("wrote gold_agent_contracts")

In [ ]:
# ---------------------------------------------------------------- self-check
# Every claim the agent can make must trace to a supplied ID, so verify that before the
# agent ever sees the envelope. Catching it here is cheap; catching it in a gate after
# generation means a wasted run, and catching it in a review means it reached a reader.
problems = []
for contract in contracts:
    patient_id = contract["patient"]["patient_id"]
    if not contract["evidence"]:
        problems.append(f"{patient_id}: no evidence, but the patient was surfaced")
    if not contract["criteria"]:
        problems.append(f"{patient_id}: no criteria, but the patient was surfaced")
    ids = [item["evidence_id"] for item in contract["evidence"]]
    if len(ids) != len(set(ids)):
        problems.append(f"{patient_id}: duplicate evidence_id")
    for item in contract["evidence"]:
        if not item["evidence_text"] or not item["evidence_id"]:
            problems.append(f"{patient_id}: evidence item missing text or id")
    if contract["patient"]["referral_state"] != "indicators_present":
        problems.append(f"{patient_id}: contract built for a patient not surfaced")

print(f"contracts checked: {len(contracts)}")
if problems:
    for problem in problems[:15]:
        print("  ", problem)
    raise ValueError(f"{len(problems)} contract problem(s)")
print("all contracts are internally consistent")

example = contracts[0]
print("\n--- example contract ---")
print(json.dumps(example, indent=2)[:1600])